In [1]:
import fitz
import io
from PIL import Image
import pytesseract
from docx import Document
import pandas as pd
import os

import fitz: This imports PyMuPDF, an incredibly fast library used here to handle PDF parsing, page rendering, and text extraction.

import io: Provides Python’s in-memory stream tools. It allows you to treat binary data (like raw image bytes) as if it were an open file.

from PIL import Image: Imports the Pillow library, which is the standard Python tool for opening, manipulating, and saving images.

import pytesseract: A wrapper for Google's Tesseract-OCR Engine. It reads text embedded inside images.

from docx import Document: Imports python-docx to read and manipulate Microsoft Word (.docx) files.

import pandas as pd: Imports the powerful Pandas data analysis library, aliased as pd. It is used here to handle tabular data from Excel and CSV files.

import os: Python's built-in operating system interface, used here to check if file paths actually exist on your hard drive

In [ ]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

What it does: Because Tesseract is an external command-line program (not a native Python library), Windows users have to explicitly tell Python where the tesseract.exe engine is installed. The r before the string marks it as a "raw string" so backslashes aren't treated as escape characters.

In [ ]:
def extract_pdf(path):
    doc = fitz.open(path)

    text = ""

    for page in doc:
        page_text = page.get_text()

        if page_text.strip():
            text += page_text
        else:
            pix = page.get_pixmap(dpi=300)
            image = Image.open(io.BytesIO(pix.tobytes("png")))
            text += pytesseract.image_to_string(image)

    return text

doc = fitz.open(path): Opens the PDF file using PyMuPDF.

text = "": Initializes an empty string to accumulate all the extracted text from the document.
for page in doc:: Loops through every single page in the PDF document sequentially.

page_text = page.get_text(): Tries to extract digital text directly embedded in the page layers.
if page_text.strip():: Checks if the extracted text contains actual characters (stripping away whitespaces/newlines). If it does, it's a digital PDF page.

text += page_text: Appends that text directly to our master text variable.

else:: If page_text is empty, it means this page is likely a scanned image or a photo.

pix = page.get_pixmap(dpi=300): Converts ("renders") the PDF page into a high-resolution pixel image (at 300 DPI, which is ideal for OCR accuracy).

image = Image.open(...): Converts the raw image bytes into a format Pillow (PIL) can work with, using io.BytesIO to simulate saving and reopening it without writing a file to your hard drive.

text += pytesseract.image_to_string(image): Passes that rendered page image to Tesseract to read the text out of the image, then appends it to the master string.